# Synthetic Data Generation for RAG Evaluation

Session 1 built a vector RAG application over a cat health guideline PDF. This
session creates an evaluation dataset for that application and uses the dataset
to compare two retrieval configurations. All generation, embedding, RAG, and
judge requests are routed through Vercel AI Gateway.

The workflow is:

~~~text
corpus -> knowledge graph -> synthetic examples -> human review
       -> LangSmith dataset -> baseline and candidate experiments
~~~

Synthetic examples reduce the cost of getting started, but generated references
are not automatically ground truth. We will inspect and curate them before using
them as evaluation targets.

> This is an educational cat health exercise, not veterinary advice. Generated
> questions and answers must not be used to diagnose, prescribe, or replace a
> veterinarian.

## Learning Outcomes

By the end of this notebook, you will be able to:

- Explain how Ragas builds a knowledge graph for test data generation.
- Distinguish single-hop specific, multi-hop specific, and multi-hop abstract queries.
- Generate and review synthetic questions, reference contexts, and reference answers.
- Route generation, embeddings, RAG, and judge calls through Vercel AI Gateway.
- Upload reviewed examples to a LangSmith dataset.
- Evaluate answer correctness, answer groundedness, and retrieval relevance.
- Run a controlled RAG experiment that changes one variable at a time.

## Table of Contents

- **Breakout Room #1: Synthetic Test Data with Ragas**
  - Task 1: Environment Setup
  - Task 2: Load the Cat Health Corpus
  - Task 3: Build and Enrich a Knowledge Graph
  - Task 4: Inspect the Query Distribution
  - Task 5: Generate and Inspect a Synthetic Test Set
  - Activity #1: Review and Curate the Dataset
- **Breakout Room #2: RAG Evaluation with LangSmith**
  - Task 6: Create a LangSmith Dataset
  - Task 7: Build a Baseline RAG Application
  - Task 8: Define RAG Evaluators
  - Task 9: Run the Baseline Experiment
  - Task 10: Change One Retrieval Variable and Re-Evaluate
  - Activity #2: Compare, Diagnose, and Iterate
  - Advanced Build: Add Robustness and Adversarial Cases

---
# Breakout Room #1
## Synthetic Test Data with Ragas

Ragas uses the source corpus to create a richer representation of its topics and
relationships. Query synthesizers then select scenarios from that representation
and generate questions plus reference answers.

The knowledge graph is a generation aid. It is not the graph used by the RAG
application in Breakout Room #2.

## Task 1: Environment Setup

From the <code>05_Synthetic_Data_Generation_for_RAG_Evals</code> folder:

~~~bash
uv sync
~~~

Then select the environment created by uv as this notebook's kernel.

Required accounts:

- Vercel AI Gateway for generation, embeddings, the RAG answer model, and judges
- LangSmith for the dataset and experiments

A direct OpenAI API key is not required. The OpenAI SDK is used only as a
protocol-compatible client pointed at Vercel AI Gateway.

The default synthetic test set is intentionally small. Ragas generation and
LLM-as-judge evaluation both make multiple model calls, so start small and scale
only after inspecting quality.

### Imports

In [1]:
from __future__ import annotations

import os
from collections import Counter
from getpass import getpass
from importlib.metadata import version
from pathlib import Path
from uuid import uuid4

import instructor
from IPython.display import display
from openai import OpenAI
from pydantic import BaseModel, field_validator

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langsmith import Client, evaluate
from openevals.llm import create_llm_as_judge
from openevals.prompts import (
    CORRECTNESS_PROMPT,
    RAG_GROUNDEDNESS_PROMPT,
    RAG_RETRIEVAL_RELEVANCE_PROMPT,
)

from ragas.embeddings import embedding_factory
from ragas.llms import llm_factory
from ragas.run_config import RunConfig
from ragas.testset import TestsetGenerator
from ragas.testset.graph import KnowledgeGraph, Node, NodeType
from ragas.testset.synthesizers import (
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer,
    SingleHopSpecificQuerySynthesizer,
    default_query_distribution,
)
from ragas.testset.transforms import (
    CustomNodeFilter,
    SummaryExtractor,
    apply_transforms,
    default_transforms_for_prechunked,
)

/Users/yuexin/Documents/cowork OS/Learning/AI Maker Space/AI Maker Space Resources/The-AI-Engineering-Certification-v1.0/05_Synthetic_Data_Generation_for_RAG_Evals/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### API Keys, Models, and Cost Controls

The notebook reads model names and budgets from environment variables so you can
tune cost without editing every cell. Vercel AI Gateway exposes an
OpenAI-compatible endpoint, so the OpenAI and LangChain clients only need a
different API key, base URL, and provider-qualified model ID.

See the [Vercel AI Gateway Python documentation](https://vercel.com/docs/ai-gateway/sdks-and-apis/python)
for the current authentication and endpoint details.

LangSmith uses <code>LANGSMITH_TRACING</code>. The older
<code>LANGCHAIN_TRACING_V2</code> name from the source notebook is no longer
needed here.

In [2]:
gateway_api_key = (
    os.environ.get("AI_GATEWAY_API_KEY")
    or os.environ.get("VERCEL_OIDC_TOKEN")
)

if not gateway_api_key:
    gateway_api_key = getpass("Vercel AI Gateway API Key: ")
    os.environ["AI_GATEWAY_API_KEY"] = gateway_api_key

if not os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_API_KEY"] = getpass("LangSmith API Key: ")

os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault(
    "LANGSMITH_PROJECT",
    "aim-session-5-synthetic-rag-evals",
)

GATEWAY_BASE_URL = os.environ.get(
    "AI_GATEWAY_BASE_URL",
    "https://ai-gateway.vercel.sh/v1",
)
GENERATOR_MODEL_NAME = os.environ.get(
    "AIM_GENERATOR_MODEL",
    "openai/gpt-5.4-mini",
)
RAG_MODEL_NAME = os.environ.get(
    "AIM_RAG_MODEL",
    "openai/gpt-5.4-mini",
)
JUDGE_MODEL_NAME = os.environ.get(
    "AIM_JUDGE_MODEL",
    "openai/gpt-5.4-mini",
)
EMBEDDING_MODEL_NAME = os.environ.get(
    "AIM_EMBEDDING_MODEL",
    "openai/text-embedding-3-small",
)
TESTSET_SIZE = int(os.environ.get("AIM_TESTSET_SIZE", "6"))
MAX_CONCURRENCY = int(os.environ.get("AIM_MAX_CONCURRENCY", "2"))

gateway_models = {
    "generator": GENERATOR_MODEL_NAME,
    "rag": RAG_MODEL_NAME,
    "judge": JUDGE_MODEL_NAME,
    "embedding": EMBEDDING_MODEL_NAME,
}
for role, model_name in gateway_models.items():
    if "/" not in model_name:
        raise ValueError(
            f"{role} model must use a provider-qualified AI Gateway ID: "
            f"{model_name!r}"
        )

print(f"Ragas: {version('ragas')}")
print(f"LangSmith: {version('langsmith')}")
print(f"AI Gateway: {GATEWAY_BASE_URL}")
print(f"Generator model: {GENERATOR_MODEL_NAME}")
print(f"RAG model: {RAG_MODEL_NAME}")
print(f"Judge model: {JUDGE_MODEL_NAME}")
print(f"Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"Synthetic examples: {TESTSET_SIZE}")
print(f"LangSmith tracing: {os.environ['LANGSMITH_TRACING']}")

Ragas: 0.4.4.dev8+g298b68274
LangSmith: 0.8.18
AI Gateway: https://ai-gateway.vercel.sh/v1
Generator model: openai/gpt-5.4-mini
RAG model: openai/gpt-5.4-mini
Judge model: openai/gpt-5.4-mini
Embedding model: openai/text-embedding-3-small
Synthetic examples: 6
LangSmith tracing: true


## Task 2: Load the Cat Health Corpus

The corpus is the bundled 2021 AAHA/AAFP Feline Life Stage Guidelines PDF used
in Session 1. <code>PyPDFLoader</code> extracts one LangChain document per page,
including page metadata that survives later chunking.

This gives the generator multiple related units to connect:

- hydration and urinary signs
- preventive care and senior care
- dental pain and behavior changes
- monitoring and emergency escalation

In [3]:
corpus_path = Path("data/cat_health_guidelines.pdf")

if not corpus_path.exists():
    raise FileNotFoundError(
        f"Expected the course corpus at {corpus_path.resolve()}"
    )

pdf_loader = PyPDFLoader(str(corpus_path))
source_documents = pdf_loader.load()
source_documents = [
    document
    for document in source_documents
    if len(document.page_content.strip()) >= 200
]

for index, document in enumerate(source_documents):
    page_number = int(document.metadata.get("page", index)) + 1
    document.metadata.update(
        {
            "source": corpus_path.name,
            "document_type": "feline_life_stage_guidelines",
            "page_number": page_number,
        }
    )

print(f"Loaded {len(source_documents)} text-containing PDF pages")
for document in source_documents[:5]:
    page_number = document.metadata["page_number"]
    print(
        f"- page {page_number}: "
        f"{len(document.page_content)} characters"
    )

Loaded 20 text-containing PDF pages
- page 1: 4913 characters
- page 2: 2084 characters
- page 3: 5955 characters
- page 6: 5673 characters
- page 7: 3588 characters


Inspect one PDF page and its metadata. The metadata is useful for debugging,
trace inspection, and explaining where a retrieved chunk came from.

In [4]:
sample_document = source_documents[0]

print(sample_document.metadata)
print()
print(sample_document.page_content[:800])

{'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'source': 'cat_health_guidelines.pdf', 'total_pages': 22, 'page': 0, 'page_label': '1', 'document_type': 'feline_life_stage_guidelines', 'page_number': 1}

VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are published simultaneously in theJournal of Feline Medicine and
Surgery(volume 23, issue 3, pages 211–233, DOI: 10.1177

## Task 3: Build and Enrich a Knowledge Graph

The unrolled workflow makes the generation stages visible:

1. Treat each text-containing PDF page as a pre-chunked Ragas node.
2. Run Ragas extractors, embeddings, and relationship builders.
3. Save the graph so expensive enrichment can be reused.

Ragas remains responsible for graph enrichment and synthetic generation. The
newer pinned Ragas build exposes an official Instructor mode parameter, so its
LLM factory can use AI Gateway tool calls directly without custom wrappers.

In [5]:
gateway_client = OpenAI(
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
)

generator_llm = llm_factory(
    GENERATOR_MODEL_NAME,
    provider="openai",
    client=gateway_client,
    mode=instructor.Mode.TOOLS,
    max_tokens=4096,
)
# Provider-qualified Gateway IDs bypass Ragas's GPT-5 parameter detection.
# Keep only the token limit supported by the Gateway route. max_retries is
# consumed locally by Instructor and is not sent to AI Gateway.
generator_llm.model_args = {
    "max_tokens": 4096,
    "max_retries": 3,
}

generator_embeddings = embedding_factory(
    "openai",
    model=EMBEDDING_MODEL_NAME,
    client=gateway_client,
)

ragas_run_config = RunConfig(
    timeout=180,
    max_retries=3,
    max_wait=30,
    max_workers=MAX_CONCURRENCY,
)

/var/folders/7g/_j2w5svx1pg5f58hd2chyy3m0000gn/T/ipykernel_85670/1199448542.py:21: DeprecationWarning: Importing embedding_factory from ragas.embeddings is deprecated. Import directly from ragas.embeddings.base or use modern providers: from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = embedding_factory(


Before building the graph, make one small structured-output request through
Ragas. This catches gateway authentication, model availability, and tool-calling
incompatibilities without waiting for every PDF page to retry.

In [6]:
class GatewayToolCallCheck(BaseModel):
    status: str


class NonEmptySummary(BaseModel):
    text: str

    @field_validator("text")
    @classmethod
    def require_text(cls, value):
        value = value.strip()
        if not value:
            raise ValueError("summary text cannot be empty")
        return value


gateway_check = generator_llm.generate(
    "Use the required tool with a short, non-empty status message.",
    GatewayToolCallCheck,
)
if not gateway_check.status.strip():
    raise RuntimeError("AI Gateway returned an empty tool-call check")

print(f"AI Gateway tool-based structured output: {gateway_check.status}")

AI Gateway tool-based structured output: ok


In [7]:
def build_prechunked_knowledge_graph(chunks):
    return KnowledgeGraph(
        nodes=[
            Node(
                type=NodeType.CHUNK,
                properties={
                    "page_content": chunk.page_content,
                    "document_metadata": dict(chunk.metadata),
                },
            )
            for chunk in chunks
            if chunk.page_content.strip()
        ]
    )


generation_chunks = list(source_documents)
knowledge_graph = build_prechunked_knowledge_graph(generation_chunks)

print(f"Ragas input chunks: {len(generation_chunks)}")
print(knowledge_graph)

Ragas input chunks: 20
KnowledgeGraph(nodes: 20, relationships: 0)


In [32]:
# [Student addition] Define artifact paths early so the expensive enrichment and
# generation cells below can check whether saved outputs already exist before
# re-running steps that took several hours.
artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(exist_ok=True)
knowledge_graph_path = artifacts_dir / "cat_health_knowledge_graph.json"
testset_path = artifacts_dir / "cat_health_synthetic_testset.jsonl"

### Apply Ragas Transforms

Because the PDF loader already gives us coherent page-level chunks, use Ragas'
built-in pre-chunked transform pipeline. It skips headline extraction and
splitting, then applies Ragas summaries, embeddings, themes, named entities,
and relationship builders directly to the PDF pages. The parent-child node
filter is omitted because these page chunks intentionally have no parent nodes.
A non-empty output constraint makes Instructor retry blank Ragas summaries before
the built-in embedding transform runs.

In [33]:
# [Student addition] Skip the 2-hour graph enrichment if a saved graph already exists.
if knowledge_graph_path.exists():
    knowledge_graph = KnowledgeGraph.load(str(knowledge_graph_path))
    print(f"Loaded existing knowledge graph — skipping enrichment.")
    print(knowledge_graph)
else:
    knowledge_graph = build_prechunked_knowledge_graph(generation_chunks)
    transforms = [
        transform
        for transform in default_transforms_for_prechunked(
            llm=generator_llm,
            embedding_model=generator_embeddings,
        )
        if not isinstance(transform, CustomNodeFilter)
    ]

    summary_transform = next(
        transform
        for transform in transforms
        if isinstance(transform, SummaryExtractor)
    )
    summary_transform.prompt.output_model = NonEmptySummary

    print("Ragas transform pipeline:")
    for transform in transforms:
        nested = getattr(transform, "transformations", None)
        if nested is None:
            print(f"- {type(transform).__name__}")
        else:
            names = ", ".join(type(item).__name__ for item in nested)
            print(f"- Parallel({names})")

    for transform in transforms:
        apply_transforms(
            knowledge_graph,
            transform,
            run_config=ragas_run_config,
        )
        if isinstance(transform, SummaryExtractor):
            empty_summary_nodes = [
                node
                for node in knowledge_graph.nodes
                if not str(node.get_property("summary") or "").strip()
            ]
            if empty_summary_nodes:
                raise RuntimeError(
                    "Ragas did not produce non-empty summaries for "
                    f"{len(empty_summary_nodes)} PDF chunks"
                )

    print(knowledge_graph)

Loaded existing knowledge graph — skipping enrichment.
KnowledgeGraph(nodes: 20, relationships: 75)


Inspect the graph at a high level. Different Ragas versions may add different
properties, so the notebook avoids depending on one exact internal schema.

In [9]:
node_type_counts = Counter(str(node.type) for node in knowledge_graph.nodes)

print("Node types:")
for node_type, count in node_type_counts.items():
    print(f"- {node_type}: {count}")

print(f"Relationships: {len(knowledge_graph.relationships)}")

for index, node in enumerate(knowledge_graph.nodes[:3], start=1):
    property_names = sorted(node.properties.keys())
    print(f"Node {index} properties: {property_names}")

Node types:
- NodeType.CHUNK: 20
Relationships: 75
Node 1 properties: ['document_metadata', 'entities', 'page_content', 'summary', 'summary_embedding', 'themes']
Node 2 properties: ['document_metadata', 'entities', 'page_content', 'summary', 'summary_embedding', 'themes']
Node 3 properties: ['document_metadata', 'entities', 'page_content', 'summary', 'summary_embedding', 'themes']


### Save and Reload the Graph

Generated artifacts go in the ignored <code>artifacts/</code> folder so running
the notebook does not add large, machine-generated files to the assignment diff.

In [ ]:
artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(exist_ok=True)

knowledge_graph_path = artifacts_dir / "cat_health_knowledge_graph.json"

# [Student addition] Only save if we just built the graph; skip if it was loaded from disk.
if not knowledge_graph_path.exists():
    knowledge_graph.save(str(knowledge_graph_path))
    print(f"Saved graph to {knowledge_graph_path}")
else:
    print(f"Graph already exists at {knowledge_graph_path} — skipping save.")

loaded_knowledge_graph = KnowledgeGraph.load(str(knowledge_graph_path))
print(loaded_knowledge_graph)

#### ❓ Question #1

What information did the Ragas graph transforms add beyond the original text?

- Ragas graph extracts entities as nodes, and then link these entities via either next (sequential) or similar (semantic) relationship.

And why are the two relationship types important for multi-hop questions?

- because multi-hop questions arequire abstraction of different entities and their relationship, to be able to connects one dots versus another (aka hop). This is where Ragas graph is utilized. The alternative is dense retrieval, which is a chunk of text that is unaware of explicit relationships across different entities.

## Task 4: Inspect the Query Distribution

Ragas can synthesize several kinds of questions:

| Query type | What it tests |
|---|---|
| Single-hop specific | Retrieve one concrete fact or recommendation from one context |
| Multi-hop specific | Combine concrete details from multiple related contexts |
| Multi-hop abstract | Connect broader themes or concepts across contexts |

The distribution is part of the evaluation specification. It determines which
behaviors are common in the generated dataset.

In [11]:
query_distribution = default_query_distribution(
    generator_llm,
    kg=loaded_knowledge_graph,
)

print("Available query synthesizers:")
for synthesizer, weight in query_distribution:
    print(f"- {synthesizer.name}: {weight:.0%}")

distribution_total = sum(weight for _, weight in query_distribution)
print(f"Distribution total: {distribution_total:.2f}")

Available query synthesizers:
- single_hop_specific_query_synthesizer: 33%
- multi_hop_abstract_query_synthesizer: 33%
- multi_hop_specific_query_synthesizer: 33%
Distribution total: 1.00


### Define a Custom Distribution

The default is a sensible starting point, but the mix should reflect the
application behavior you care about. This example emphasizes concrete
single-hop questions while preserving coverage of both multi-hop styles.

Adjust the weights below and assign
<code>query_distribution = custom_query_distribution</code> before Task 5 if
you want the generated dataset to use your mix. We define the distribution here
without generating a second test set, which keeps the worked notebook's cost
bounded.

The default helper filters out synthesizers that the enriched graph cannot
support. If a custom multi-hop run reports that no matching relationships exist,
inspect the graph and use only the synthesizers listed by the default distribution.

In [12]:
custom_query_distribution = [
    (
        SingleHopSpecificQuerySynthesizer(llm=generator_llm),
        0.50,
    ),
    (
        MultiHopSpecificQuerySynthesizer(llm=generator_llm),
        0.30,
    ),
    (
        MultiHopAbstractQuerySynthesizer(llm=generator_llm),
        0.20,
    ),
]

assert abs(
    sum(weight for _, weight in custom_query_distribution) - 1.0
) < 1e-9

for synthesizer, weight in custom_query_distribution:
    print(f"- {synthesizer.name}: {weight:.0%}")

- single_hop_specific_query_synthesizer: 50%
- multi_hop_specific_query_synthesizer: 30%
- multi_hop_abstract_query_synthesizer: 20%


#### ❓ Question #2

Describe the three query types in your own words. Which type do you expect to be
hardest for a basic dense-retrieval RAG application, and why?

##### ✅ Answer

**Single-hop specific:** Tests whether the RAG system can retrieve and return one concrete fact or recommendation from a single source chunk.

**Multi-hop specific:** Tests whether the system can combine precise details drawn from two or more related chunks into a single coherent answer.

**Multi-hop abstract:** Tests whether the system can reason across broader themes or concepts that are distributed across the corpus without an obvious keyword match.

---

**Hardest type:** Multi-hop abstract is the hardest for basic dense-retrieval RAG.

**Why:** A single embedding query vector can't simultaneously represent two thematically related but worded differently concepts, so the retriever fails to surface all the chunks needed before the answer can even be assembled.

## Task 5: Generate and Inspect a Synthetic Test Set

Each generated row contains:

- <code>user_input</code>: the synthetic question
- <code>reference_contexts</code>: source context selected by the generator
- <code>reference</code>: a generated reference answer
- <code>synthesizer_name</code>: the query strategy that produced the row

The reference is generated from selected source context. It is useful, but it
still needs review for accuracy, clarity, safety, and usefulness.

In [ ]:
import pandas as pd

# [Student addition] Skip expensive test set generation if saved results already exist.
if testset_path.exists():
    testset_df = pd.read_json(testset_path, orient="records", lines=True)
    print(f"Loaded existing test set from {testset_path}")
    display(
        testset_df[
            [
                "user_input",
                "reference",
                "synthesizer_name",
            ]
        ]
    )
else:
    testset_generator = TestsetGenerator(
        llm=generator_llm,
        embedding_model=generator_embeddings,
        knowledge_graph=loaded_knowledge_graph,
    )

    synthetic_testset = testset_generator.generate(
        testset_size=TESTSET_SIZE,
        query_distribution=query_distribution,
        run_config=ragas_run_config,
    )

    testset_df = synthetic_testset.to_pandas()

    display(
        testset_df[
            [
                "user_input",
                "reference",
                "synthesizer_name",
            ]
        ]
    )

In [ ]:
testset_path = artifacts_dir / "cat_health_synthetic_testset.jsonl"

# [Student addition] Only save if we just generated the test set; skip if it was loaded from disk.
if not testset_path.exists():
    testset_df.to_json(
        testset_path,
        orient="records",
        lines=True,
        force_ascii=False,
    )
    print(f"Saved candidate examples to {testset_path}")
else:
    print(f"Test set already exists at {testset_path} — skipping save.")

print("Examples by synthesizer:")
print(testset_df["synthesizer_name"].value_counts())

### Abstracted Ragas Alternative

The graph-first path above makes each Ragas stage inspectable and lets you save
the enriched graph before generation. Ragas also provides a one-call helper for
content that is already chunked:

~~~python
quick_generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
)
quick_testset = quick_generator.generate_with_chunks(
    chunks=generation_chunks,
    testset_size=TESTSET_SIZE,
    transforms=transforms,
    run_config=ragas_run_config,
)
~~~

This alternative is shown rather than executed so the notebook does not repeat
the same billable graph enrichment and test-set generation.

#### ❓ Question #3

What are the tradeoffs between the unrolled and one-call Ragas generation paths?
When would you choose each one?

##### ✅ Answer

**Tradeoffs:**

- **Unrolled** breaks the pipeline into explicit steps — you can inspect the graph after each stage, customise individual transforms (like the notebook's custom summary prompt), save the enriched graph to disk so expensive LLM calls don't need to re-run, and pinpoint exactly which step failed if something goes wrong.
- **One-call** (`generate_with_chunks`) bundles all steps into a single function — less code, but the intermediate graph is invisible, unsaveable, and uncustomisable.

**When to choose each:**

- Use **unrolled** when building or debugging a new pipeline, when you want to reuse the enriched graph across multiple runs, or when you need to customise a specific transform.
- Use **one-call** when the pipeline is already validated and you just need a quick result without caring about intermediate state.

## 🏗️ Activity #1: Review and Curate the Dataset

Review every generated row before uploading it.

For each example, check:

1. Is the question answerable from the reference contexts?
2. Is the reference answer fully supported by those contexts?
3. Is the wording natural for a plausible user?
4. Does the example duplicate another row?
5. Does it preserve the corpus's medical-safety boundaries?

Requirements:

- Remove or repair at least one weak example, if one exists.
- Record why you kept, edited, or removed it.
- Keep the synthesizer name in metadata so you can diagnose query-type failures.

In [20]:
# Activity #1 workspace
approved_testset_df = testset_df.drop(index=[0]).reset_index(drop=True)
review_status = "student_reviewed"

display(
    approved_testset_df[
        [
            "user_input",
            "reference_contexts",
            "reference",
            "synthesizer_name",
        ]
    ]
)

,user_input,reference_contexts,reference,synthesizer_name
0,Why is a life stage assessment important durin...,[Introduction\nThe feline patient ’s life stag...,A life stage assessment is important during ea...,single_hop_specific_query_synthesizer
1,In a cat with changes in grooming habits that ...,[<1-hop>\n\nDetecting signs of pain or anxiety...,Changes in grooming habits may signal a dermat...,multi_hop_abstract_query_synthesizer
2,In the context of feline health and nutrition ...,"[<1-hop>\n\n90. Wichert B, Muller L, Gebert S,...",The guidelines emphasize that preventive care ...,multi_hop_abstract_query_synthesizer
3,"For mature adult and senior cats, how do the d...",[<1-hop>\n\ngood starting point is to calculat...,Mature adult and senior cats have changing die...,multi_hop_specific_query_synthesizer
4,How do the AAHA/AAFP Feline Life Stage Guideli...,[<1-hop>\n\nthan observed or learned. 44 Impor...,The 2021 AAHA/AAFP Feline Life Stage Guideline...,multi_hop_specific_query_synthesizer


### 📝 Activity #1 Notes

**Example removed:** Row 0 — "Who is Jessica Quimby in the 2021 AAHA/AAFP Feline Life Stage Guidelines?"

**Decision:** Removed.

**Reason:** No cat owner would ever ask who authored the guidelines. The question tests document metadata rather than cat health knowledge, making it useless as a RAG evaluation example. A RAG system could answer it correctly and score well while being completely wrong on practical health questions.

**Remaining 5 examples reviewed:**
- Row 1 [single-hop]: kept — practical, answerable, well-grounded. Minor artificial phrasing ("in the United States") but doesn't affect the answer.
- Row 2 [multi-hop abstract]: kept — covers a realistic owner concern spanning grooming, parasites, and zoonotic risk across multiple chunks.
- Row 3 [multi-hop abstract]: kept — connects dental, parasite, and vaccination guidance in a way that reflects real preventive care decisions.
- Row 4 [multi-hop specific]: kept — strong example on dietary and hydration needs for mature/senior cats. Clear and practical.
- Row 5 [multi-hop specific]: kept — kitten first exam and socialization question is valid, wording slightly convoluted but answerable.

**Any safety or grounding issue found:** None. All remaining examples preserve the corpus's medical-safety boundaries and do not ask for diagnoses or prescriptions.

---
# Breakout Room #2
## RAG Evaluation with LangSmith

We will upload the reviewed examples, build one RAG application, and evaluate two
retrieval settings against the same dataset and judges.

Keeping the dataset and evaluators fixed makes the application change easier to
interpret.

## Task 6: Create a LangSmith Dataset

The dataset stores the question as input and the reviewed synthetic answer plus
reference contexts as outputs. Query type and provenance remain metadata.

A unique suffix prevents accidental duplication when the whole notebook is rerun.
For a long-lived team dataset, use a stable name and manage versions deliberately.

In [19]:
def as_string_list(value) -> list[str]:
    if value is None:
        return []
    if isinstance(value, list):
        return [str(item) for item in value]
    if hasattr(value, "tolist"):
        converted = value.tolist()
        if isinstance(converted, list):
            return [str(item) for item in converted]
    return [str(value)]


if review_status != "student_reviewed":
    raise ValueError(
        "Complete Activity #1, curate approved_testset_df, and set "
        "review_status = 'student_reviewed' before uploading."
    )


langsmith_client = Client()
dataset_name = (
    "aim-session-5-cat-health-synthetic-"
    f"{uuid4().hex[:8]}"
)

langsmith_dataset = langsmith_client.create_dataset(
    dataset_name=dataset_name,
    description=(
        "Ragas-generated questions for the AI Makerspace "
        "cat health RAG lesson."
    ),
    metadata={
        "session": 5,
        "source": "ragas",
        "corpus": str(corpus_path),
    },
)

langsmith_examples = []
for _, row in approved_testset_df.iterrows():
    langsmith_examples.append(
        {
            "inputs": {
                "question": str(row["user_input"]),
            },
            "outputs": {
                "answer": str(row["reference"]),
                "reference_contexts": as_string_list(
                    row["reference_contexts"]
                ),
            },
            "metadata": {
                "synthesizer_name": str(row["synthesizer_name"]),
                "synthetic_reference": True,
                "review_status": review_status,
            },
        }
    )

langsmith_client.create_examples(
    dataset_id=langsmith_dataset.id,
    examples=langsmith_examples,
)

print(f"Created dataset: {dataset_name}")
print(f"Examples uploaded: {len(langsmith_examples)}")

Created dataset: aim-session-5-cat-health-synthetic-6beebe51
Examples uploaded: 5


#### ❓ Question #4

Why is it useful to keep `synthesizer_name`,
`synthetic_reference`, and review status as metadata instead of
discarding them after upload?

##### ✅ Answer

- **`synthesizer_name`** — lets you slice evaluation failures by query type. If the RAG system scores poorly, you can filter to see whether it's failing specifically on multi-hop abstract questions vs. single-hop ones. Without this, a low average score tells you nothing about where to fix.

- **`synthetic_reference`** — flags that the reference answer was generated by an LLM (Ragas), not written by a human expert. The correctness evaluator compares the RAG answer against the reference, so if the reference is wrong, the whole evaluation is wrong. This flag reminds you to check the reference itself — not just the RAG output — when a score looks unexpectedly low.

- **`review_status`** — tells you whether a human reviewed the example before it was used as a benchmark. An unreviewed example is less trustworthy as ground truth, so a low score against it means less than a low score against a human-reviewed one.

Together, these fields let you distinguish between a bad RAG application and a bad dataset — without them, a low score is uninterpretable.

## Task 7: Build a Baseline RAG Application

The baseline uses the same PDF corpus, recursive character chunks, embeddings
and chat generation through Vercel AI Gateway, in-memory Qdrant, and a
context-only answer prompt.

The target returns both the answer and the retrieved contexts. Returning
intermediate retrieval output makes it possible to evaluate retrieval relevance
and answer groundedness without reconstructing hidden steps.

In [21]:
rag_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=75,
)
rag_documents = rag_splitter.split_documents(source_documents)

rag_embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL_NAME,
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
    check_embedding_ctx_length=False,
)
vector_store = QdrantVectorStore.from_documents(
    documents=rag_documents,
    embedding=rag_embeddings,
    location=":memory:",
    collection_name=f"cat_health_eval_{uuid4().hex[:8]}",
)

print(f"Source PDF pages: {len(source_documents)}")
print(f"RAG chunks: {len(rag_documents)}")

Source PDF pages: 20
RAG chunks: 255


In [22]:
RAG_SYSTEM_PROMPT = """You are an educational cat health assistant.

Answer the question using only the retrieved context.
If the context is insufficient, say that the corpus does not provide enough
information.

Do not diagnose, prescribe treatment, or present the response as a substitute
for a veterinarian. Clearly preserve any urgent-care guidance found in the
context.

Retrieved context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", RAG_SYSTEM_PROMPT),
        ("human", "{question}"),
    ]
)
rag_llm = ChatOpenAI(
    model=RAG_MODEL_NAME,
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
)
answer_chain = rag_prompt | rag_llm | StrOutputParser()

In [23]:
def format_retrieved_document(document) -> str:
    page_number = document.metadata.get("page_number", "unknown")
    source = document.metadata.get("source", "course corpus")
    return (
        f"Page: {page_number}\n"
        f"Source: {source}\n"
        f"{document.page_content}"
    )


def make_rag_target(retrieval_k: int):
    retriever = vector_store.as_retriever(
        search_kwargs={"k": retrieval_k}
    )

    def target(inputs: dict) -> dict:
        question = inputs["question"]
        retrieved_documents = retriever.invoke(question)
        contexts = [
            format_retrieved_document(document)
            for document in retrieved_documents
        ]
        answer = answer_chain.invoke(
            {
                "question": question,
                "context": "\n\n".join(contexts),
            }
        )
        return {
            "answer": answer,
            "contexts": contexts,
            "retrieval_k": retrieval_k,
        }

    target.__name__ = f"cat_health_rag_k_{retrieval_k}"
    return target

In [24]:
baseline_retrieval_k = 3
baseline_target = make_rag_target(baseline_retrieval_k)

spot_check_question = (
    "What components should be considered during a feline wellness visit?"
)
baseline_spot_check = baseline_target(
    {"question": spot_check_question}
)

print(baseline_spot_check["answer"])
print()
print("Retrieved contexts:")
for context in baseline_spot_check["contexts"]:
    print("---")
    print(context[:700])

The retrieved context says a feline wellness visit should consider these components:

- Nutritional and environmental needs  
- Elimination  
- Nutrition and weight management  
- Oral health  
- Parasite control  
- Vaccination  
- Zoonoses and human safety  
- Diagnostics  

It also notes additional important topics such as:

- Feline-friendly handling practices  
- Overcoming barriers to examination visits  
- Environmental enrichment  
- Understanding feline behavior  
- Practice team training  
- Client education  

The corpus does not provide more detailed breakdowns beyond these components.

Retrieved contexts:
---
Page: 1
Source: cat_health_guidelines.pdf
lifelong feline healthcare strategy. The guidelines include a comprehensive table on the components of a feline wellness visit that
provides a framework for systematically implementing an individualized life stage approach to fe line healthcare. Included are
recommendations for managing the most critical health-related factors

## Task 8: Define RAG Evaluators

We will evaluate three different relationships:

| Metric | Comparison |
|---|---|
| Answer correctness | Generated answer vs reviewed reference answer |
| Answer groundedness | Generated answer vs contexts retrieved during that run |
| Retrieval relevance | Retrieved contexts vs input question |

These can disagree. A fluent answer can be correct but unsupported by its retrieved
context, or well grounded in context that does not answer the question.

OpenEvals provides reusable prompts, while the small wrapper functions map our
application's dictionary keys into those prompts.

In [25]:
gateway_judge_llm = ChatOpenAI(
    model=JUDGE_MODEL_NAME,
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
)

correctness_judge = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,
    feedback_key="answer_correctness",
    judge=gateway_judge_llm,
    continuous=True,
)
groundedness_judge = create_llm_as_judge(
    prompt=RAG_GROUNDEDNESS_PROMPT,
    feedback_key="answer_groundedness",
    judge=gateway_judge_llm,
    continuous=True,
)
retrieval_relevance_judge = create_llm_as_judge(
    prompt=RAG_RETRIEVAL_RELEVANCE_PROMPT,
    feedback_key="retrieval_relevance",
    judge=gateway_judge_llm,
    continuous=True,
)

In [26]:
def answer_correctness(
    inputs: dict,
    outputs: dict,
    reference_outputs: dict,
) -> dict:
    return correctness_judge(
        inputs=inputs["question"],
        outputs=outputs["answer"],
        reference_outputs=reference_outputs["answer"],
    )


def answer_groundedness(
    outputs: dict,
) -> dict:
    return groundedness_judge(
        context=outputs["contexts"],
        outputs=outputs["answer"],
    )


def retrieval_relevance(
    inputs: dict,
    outputs: dict,
) -> dict:
    return retrieval_relevance_judge(
        inputs=inputs["question"],
        context=outputs["contexts"],
    )


rag_evaluators = [
    answer_correctness,
    answer_groundedness,
    retrieval_relevance,
]

#### ❓ Question #5

Give one example where answer correctness and groundedness could disagree. What
would that disagreement tell you to inspect in the trace?

##### ✅ Answer

**High correctness, low groundedness:** The RAG answer correctly describes senior cat hydration needs and matches the reference answer, but the retrieved chunks were about kitten vaccination — so the answer wasn't actually supported by what was retrieved. This means the model used its own training knowledge rather than the context, which is a hallucination risk. In the trace, you would inspect the retrieved chunks to confirm they didn't contain the answer, then check whether the model cited or ignored them.

**Low correctness, high groundedness:** The RAG answer is fully supported by the retrieved chunks, but those chunks covered parasite control rather than the nutrition question that was asked. The model faithfully summarised what it retrieved, but retrieval pulled the wrong context entirely. In the trace, you would inspect the retrieved chunk content and similarity scores to diagnose why the wrong chunks ranked highest — pointing to a retrieval problem, not a generation problem.

In both cases, the disagreement tells you which part of the pipeline to fix: generation when correctness is low despite good retrieval, and retrieval when groundedness is low despite a correct-looking answer.

## Task 9: Run the Baseline Experiment

LangSmith runs the target once for each dataset example, applies all evaluators,
and groups the traces under one experiment.

After the run, open the experiment URL and inspect individual failures. Aggregate
scores alone do not explain whether the problem came from the generated dataset,
retrieval, prompting, or the judge.

In [27]:
baseline_results = evaluate(
    baseline_target,
    data=dataset_name,
    evaluators=rag_evaluators,
    experiment_prefix="cat-health-rag-baseline-k3",
    description=(
        "Baseline cat health RAG with 500-character chunks "
        "and retrieval k=3."
    ),
    metadata={
        "chunk_size": 500,
        "chunk_overlap": 75,
        "retrieval_k": baseline_retrieval_k,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "rag_model": RAG_MODEL_NAME,
        "judge_model": JUDGE_MODEL_NAME,
        "ai_gateway_base_url": GATEWAY_BASE_URL,
    },
    max_concurrency=MAX_CONCURRENCY,
)

print(f"Baseline experiment: {baseline_results.experiment_name}")

View the evaluation results for experiment: 'cat-health-rag-baseline-k3-0516be63' at:
https://smith.langchain.com/o/00ad3b5e-4f38-484c-8258-38fa12b60838/datasets/a318b2b6-c3e6-4fb8-a2dd-7fb13fee93c0/compare?selectedSessions=7b2a2a42-c904-4ba3-b9c5-be45efbbdd11




5it [00:20,  4.17s/it]

Baseline experiment: cat-health-rag-baseline-k3-0516be63


### Baseline Inspection Notes

- **Lowest-scoring example:** "How do the AAHA/AAFP Feline Life Stage Guidelines, plus the kitten socialization and training advice, explain what should be done at a kitten's first exam so it can grow into a cat that is easier for life stage based preventive care?" (multi-hop specific)
- **Metrics that failed:** Answer correctness (0.20) and retrieval relevance (0.35) — both significantly below the other examples.
- **Was the synthetic reference valid?** Yes — the question is reasonable and the reference answer is grounded in the corpus. The problem is not the dataset.
- **Was the retrieved context relevant and sufficient?** No — retrieval relevance of 0.35 indicates k=3 did not surface the right chunks. This multi-hop question spans both kitten life stage guidance and socialization advice, and a single embedding query at k=3 was not enough to capture both topics simultaneously.
- **Did the answer add unsupported information?** No — groundedness was 0.95, meaning the model faithfully answered from whatever it retrieved. The issue is that retrieval pulled the wrong chunks, not that the model hallucinated.

**Overall:** This is a retrieval failure, not a generation failure — exactly the low correctness + high groundedness + low retrieval relevance pattern. Increasing k is the natural next variable to test.

| Metric | Mean score |
|---|---|
| Answer correctness | 0.67 |
| Answer groundedness | 0.94 |
| Retrieval relevance | 0.78 |

## Task 10: Change One Retrieval Variable and Re-Evaluate

The source notebook changed chunk size, embedding model, retriever settings, and
prompt style at the same time. That makes any score change hard to explain.

Here we change only retrieval depth:

~~~text
baseline:  k = 3
candidate: k = 6
~~~

The chunks, embeddings, vector store, answer model, prompt, dataset, and evaluators
remain fixed.

In [28]:
candidate_retrieval_k = 6
candidate_target = make_rag_target(candidate_retrieval_k)

candidate_spot_check = candidate_target(
    {"question": spot_check_question}
)

print(candidate_spot_check["answer"])
print()
print(
    "Retrieved context count:",
    len(candidate_spot_check["contexts"]),
)

The corpus says a feline wellness visit should consider these components:

- **Physical examination**
- **Behavior and environmental needs**
- **Elimination**
- **Nutrition and weight management**
- **Oral health**
- **Parasite control**
- **Vaccination**
- **Zoonoses and human safety**
- **Diagnostics**

It also notes other important topics for the visit, including:

- **Feline-friendly handling practices**
- **Overcoming barriers to examination visits**
- **Environmental enrichment**
- **Understanding feline behavior**
- **Practice team training**
- **Client education**

The corpus does not provide more detail in the retrieved context.

Retrieved context count: 6


In [29]:
candidate_results = evaluate(
    candidate_target,
    data=dataset_name,
    evaluators=rag_evaluators,
    experiment_prefix="cat-health-rag-candidate-k6",
    description=(
        "Candidate cat health RAG with the same index and "
        "retrieval k increased from 3 to 6."
    ),
    metadata={
        "chunk_size": 500,
        "chunk_overlap": 75,
        "retrieval_k": candidate_retrieval_k,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "rag_model": RAG_MODEL_NAME,
        "judge_model": JUDGE_MODEL_NAME,
        "ai_gateway_base_url": GATEWAY_BASE_URL,
        "changed_variable": "retrieval_k",
    },
    max_concurrency=MAX_CONCURRENCY,
)

print(f"Candidate experiment: {candidate_results.experiment_name}")

View the evaluation results for experiment: 'cat-health-rag-candidate-k6-9386bfa3' at:
https://smith.langchain.com/o/00ad3b5e-4f38-484c-8258-38fa12b60838/datasets/a318b2b6-c3e6-4fb8-a2dd-7fb13fee93c0/compare?selectedSessions=786a3010-43d4-461e-974d-04c3d483d69c




5it [00:19,  3.96s/it]

Candidate experiment: cat-health-rag-candidate-k6-9386bfa3


#### ❓ Question #6

Why is changing one variable at a time useful? If correctness improves while
retrieval relevance falls, what might the larger value of `k` be doing?

##### ✅ Answer

**Why change one variable at a time:** If you change chunk size, k, embedding model, and prompt all at once and scores go up, you don't know which change caused the improvement. You can't reproduce it deliberately, and you can't reverse just the part that hurt. One variable at a time means any score change has exactly one cause and is interpretable.

**If correctness improves while retrieval relevance falls:** This is a precision vs. recall tradeoff. k=6 casts a wider net — it retrieves more chunks, which increases the chance that the right answer is somewhere in the context (correctness improves). But the extra chunks are the next-best matches, not the best ones, so the average relevance of the retrieved set drops (retrieval relevance falls). The model is finding the right answer by drowning it in more context, not by retrieving more precisely.

## 🏗️ Activity #2: Compare, Diagnose, and Iterate

Compare the baseline and candidate experiments in LangSmith.

Requirements:

1. Record the mean score for each evaluator in both experiments.
2. Inspect at least two examples whose scores changed.
3. Decide whether <code>k=6</code> improved the application overall.
4. Choose one new variable to test: chunk size, chunk overlap, embedding model,
   prompt, or retrieval depth.
5. State your prediction before running the experiment.
6. Run a third experiment and explain the result.

Keep the reviewed dataset and evaluators fixed. If you discover that an example
itself is invalid, fix or remove the example and treat that as dataset maintenance,
not an application improvement.

In [31]:
# Activity #2 workspace — Third experiment: chunk_size=1000, k=3
#
# The baseline (k=3) and candidate (k=6) both used 500-character chunks.
# k=6 improved retrieval relevance but not correctness, suggesting the chunks
# themselves don't contain enough information. Larger chunks preserve more
# context per retrieval hit without increasing k.

student_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=75,
)
student_documents = student_splitter.split_documents(source_documents)

student_vector_store = QdrantVectorStore.from_documents(
    documents=student_documents,
    embedding=rag_embeddings,
    location=":memory:",
    collection_name=f"cat_health_eval_chunk1000_{uuid4().hex[:8]}",
)

print(f"RAG chunks (chunk_size=1000): {len(student_documents)}")

def make_rag_target_custom_store(store, retrieval_k):
    retriever = store.as_retriever(search_kwargs={"k": retrieval_k})

    def target(inputs: dict) -> dict:
        question = inputs["question"]
        retrieved_documents = retriever.invoke(question)
        contexts = [
            format_retrieved_document(document)
            for document in retrieved_documents
        ]
        answer = answer_chain.invoke(
            {
                "question": question,
                "context": "\n\n".join(contexts),
            }
        )
        return {
            "answer": answer,
            "contexts": contexts,
            "retrieval_k": retrieval_k,
        }

    target.__name__ = f"cat_health_rag_chunk1000_k_{retrieval_k}"
    return target

student_target = make_rag_target_custom_store(student_vector_store, retrieval_k=3)

student_results = evaluate(
    student_target,
    data=dataset_name,
    evaluators=rag_evaluators,
    experiment_prefix="cat-health-rag-chunk1000-k3",
    description=(
        "Third experiment: chunk size increased from 500 to 1000 characters, "
        "retrieval k kept at 3."
    ),
    metadata={
        "chunk_size": 1000,
        "chunk_overlap": 75,
        "retrieval_k": 3,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "rag_model": RAG_MODEL_NAME,
        "judge_model": JUDGE_MODEL_NAME,
        "ai_gateway_base_url": GATEWAY_BASE_URL,
        "changed_variable": "chunk_size",
    },
    max_concurrency=MAX_CONCURRENCY,
)

print(f"Third experiment: {student_results.experiment_name}")

RAG chunks (chunk_size=1000): 120
View the evaluation results for experiment: 'cat-health-rag-chunk1000-k3-7294e127' at:
https://smith.langchain.com/o/00ad3b5e-4f38-484c-8258-38fa12b60838/datasets/a318b2b6-c3e6-4fb8-a2dd-7fb13fee93c0/compare?selectedSessions=3af2838e-017a-4d68-a60f-fc7d5b6677f1




5it [00:23,  4.62s/it]

Third experiment: cat-health-rag-chunk1000-k3-7294e127


### 📝 Activity #2 Notes

**Variable changed:** Chunk size — increased from 500 to 1000 characters, keeping k=3 and all other variables fixed.

**Prediction:** Correctness should improve on multi-hop questions (especially the kitten question) because larger chunks preserve more context at chunk boundaries. Retrieval relevance may stay flat or drop slightly because larger chunks are less semantically focused.

---

**Baseline (k=3, chunk_size=500) mean scores:**
| Metric | Score |
|---|---|
| Answer correctness | 0.67 |
| Answer groundedness | 0.94 |
| Retrieval relevance | 0.78 |

**Candidate (k=6, chunk_size=500) mean scores:**
| Metric | Score |
|---|---|
| Answer correctness | 0.66 |
| Answer groundedness | 0.95 |
| Retrieval relevance | 0.86 |

**Third experiment (k=3, chunk_size=1000) mean scores:**
| Metric | Score |
|---|---|
| Answer correctness | 0.61 |
| Answer groundedness | 0.88 |
| Retrieval relevance | 0.82 |

---

**Two traces inspected:**

1. **Kitten first exam question** — the worst performer in both baseline and candidate (correctness 0.20, retrieval relevance 0.35 at k=3 and 0.62 at k=6). Retrieval relevance improved with k=6 but correctness did not move, suggesting the retrieved chunks didn't contain the full context needed even when more were retrieved. With chunk_size=1000, correctness improved slightly to 0.40 but is still very low — this question appears genuinely hard for dense retrieval regardless of chunking strategy.

2. **Grooming + parasite + zoonotic question** — correctness dropped from 0.95 (k=3) to 0.82 (k=6), despite retrieval relevance holding steady. This shows that adding more chunks can dilute the context and hurt generation quality even when retrieval itself doesn't worsen.

**Decision on k=6:** Mixed result. Retrieval relevance improved (+0.08) but correctness was flat (-0.01). Not a clear win.

**Third experiment result:** The prediction was wrong — larger chunks made things worse. Correctness dropped from 0.67 to 0.61 and groundedness from 0.94 to 0.88. Larger chunks appear to introduce noise: each retrieved chunk contains more unrelated content alongside the relevant passage, which confuses the model. The preventive care question was the clearest casualty, dropping from 0.72 to 0.30.

**Decision overall:** The baseline (k=3, chunk_size=500) remains the best configuration on correctness and groundedness. k=6 is the best on retrieval relevance but doesn't translate to better answers. The kitten multi-hop question is a persistent failure case that may require a fundamentally different approach — such as query decomposition or hybrid retrieval — rather than tuning k or chunk size alone.

**Cost or latency tradeoff:** k=6 doubles context tokens vs k=3 at no correctness gain. chunk_size=1000 halves the number of chunks (~130 vs 255) but each chunk is larger, keeping token costs similar to k=6. Neither change justifies its cost over the baseline.

## Advanced Build: Add Robustness and Adversarial Cases

Synthetic data can cover failure modes as well as happy-path questions.

Add at least three reviewed cases such as:

- A user asks for a diagnosis or medication dose that the corpus cannot support.
- A prompt-injection attempt asks the assistant to ignore its context-only policy.
- An unrelated question should trigger an insufficient-context response.
- Retrieved text contains a malicious instruction that should be treated as data,
  not as an instruction.

For each case, define the expected behavior and an evaluator that measures it.
Track normal-task performance and attack resistance separately so a system does
not appear safe merely because it refuses everything.

## Final Takeaways

- Synthetic data is a starting point for evaluation, not a replacement for human
  review or production examples.
- The knowledge graph and query distribution shape which capabilities the dataset
  measures.
- Store provenance and review metadata so failures can be traced back to the data.
- Return retrieval output from the target when retrieval and grounding matter.
- Evaluate retrieval, grounding, and answer quality separately.
- Change one application variable at a time when you want an interpretable result.